In [1]:
import pandas as pd
import numpy as np
import re

def to_snake(s: str) -> str:
    if not isinstance(s, str):
        return s
    s = s.strip().lower()
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", "_", s)
    return s

def normalize_station_key(name: str) -> str:
    """Normalize station names for robust joining across files."""
    if pd.isna(name):
        return np.nan
    s = str(name).lower().strip()
    s = re.sub(r"[^\w\s]", " ", s)
    # Drop common noise words
    noise = ["charging", "ev", "station", "charger", "fast", "slow", "dc", "ac"]
    tokens = [t for t in s.split() if t not in noise]
    s = " ".join(tokens)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def series_mode(x: pd.Series):
    """Most frequent value; returns NaN if empty."""
    if x.empty:
        return np.nan
    counts = x.value_counts(dropna=True)
    return counts.index[0] if not counts.empty else np.nan

def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km (vectorized friendly)."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c


Session Master:


In [2]:
import pandas as pd
import numpy as np

# File path
file_path = "C:\\Users\\Yash\\OneDrive\\Desktop\\Yash\\Evision\\Evision\\Files\\Cleaned dataset\\Complete_Master_Data.csv"

# Load the dataset
df = pd.read_csv(file_path)

# Function to convert column names to snake_case
def to_snake(name):
    return name.strip().lower().replace(" ", "_")

# Function to normalize station keys
def normalize_station_key(name):
    if pd.isnull(name):
        return np.nan
    return name.strip().lower().replace(" ", "_")

# Standardize all column names to snake_case
df.columns = [to_snake(c) for c in df.columns]

# Rename columns where necessary to match expected naming
rename_map = {
    "energy_consumed": "energy_consumed_kwh",
    "charging_duration": "charging_duration_hours",
    "charging_rate": "charging_rate_kw",
    "charger_type": "charger_type"  # already correct, but for completeness
}
df = df.rename(columns=rename_map)

# Ensure required columns exist, fill with NaN if missing
needed = ["energy_consumed_kwh", "charging_duration_hours", "charging_rate_kw",
          "time_of_day", "day_of_week", "charger_type", "user_type"]
for col in needed:
    if col not in df.columns:
        df[col] = np.nan

# Ensure 'station_name' exists
if "station_name" not in df.columns and "charging_station_location" in df.columns:
    df["station_name"] = df["charging_station_location"]
elif "station_name" not in df.columns:
    df["station_name"] = "unknown"

# Create 'station_key' for joining or grouping purposes
df["station_key"] = df["station_name"].apply(normalize_station_key)

# Save the cleaned dataset
output_file = "sessions_master.csv"
df.to_csv(output_file, index=False)

# Display output like you asked
print(f"{output_file} ->", df.shape)
df.head(3)

sessions_master.csv -> (314, 22)


,charging_station_id,charging_station_operator,charging_station_name,charging_station_location,latitude,longitude,city,state,vehicle_model,battery_capacity,...,charging_rate_kw,state_of_charge_start,state_of_charge_end,time_of_day,day_of_week,payment_type,user_type,charger_type,station_name,station_key
0,Station_391,ATUM,ATUM Charge - Mulund,"Mulund West, Mumbai",19.1721,72.9568,Mumbai,Maharashtra,TATA Tiago,108.463007,...,36.389181,29.371576,86.119962,Evening,Tuesday,"App, Card",Commuter,DC Fast Charger,"Mulund West, Mumbai","mulund_west,_mumbai"
1,Station_327,EV Point,EV Point - Juhu,"Juhu Tara Rd, Near Juhu Beach, Mumbai, Maharas...",19.0994,72.8269,Mumbai,Maharashtra,Hyundai Kona,50.000000,...,32.882870,83.120003,99.624328,Evening,Saturday,"App, Card",Long-Distance Traveler,Level 1,"Juhu Tara Rd, Near Juhu Beach, Mumbai, Maharas...","juhu_tara_rd,_near_juhu_beach,_mumbai,_maharas..."
2,Station_162,ChargeZone,ChargeZone - Ambernath,"Ambernath Station, Thane",19.1860,73.1891,Mumbai,Maharashtra,MG Comet,85.000000,...,26.185188,60.751781,70.796097,Evening,Friday,"App, Card",Commuter,Level 2,"Ambernath Station, Thane","ambernath_station,_thane"


Station Aggregates:

In [3]:
import pandas as pd
import numpy as np

# Load the uploaded dataset
file_path = 'sessions_master.csv'
sessions_master = pd.read_csv(file_path)

# Helper function to calculate mode safely (for any data type)
def series_mode(series):
    non_na = series.dropna()
    if non_na.empty:
        return np.nan
    mode_series = non_na.mode()
    return mode_series.iloc[0] if not mode_series.empty else np.nan

# Haversine function to calculate distances between coordinates
def haversine_km(lat1, lon1, lat2, lon2):
    from math import radians, sin, cos, sqrt, atan2
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# Build aggregates dictionary based on actual column names
agg_dict = {}
if "energy_consumed_kwh" in sessions_master.columns:
    agg_dict["energy_consumed_kwh"] = ["sum", "mean"]
if "charging_duration_hours" in sessions_master.columns:
    agg_dict["charging_duration_hours"] = "mean"
if "charging_rate_kw" in sessions_master.columns:
    agg_dict["charging_rate_kw"] = "mean"

# Group columns based on actual column names
group_cols = ["station_key"]
if "station_name" in sessions_master.columns:
    group_cols.append("station_name")
if "charging_station_operator" in sessions_master.columns:
    group_cols.append("charging_station_operator")

# Aggregate
station_agg = (
    sessions_master
    .groupby(group_cols, dropna=False)
    .agg(agg_dict)
    .reset_index()
)

# Flatten multi-index columns
station_agg.columns = ["_".join(col).rstrip("_") if isinstance(col, tuple) else col for col in station_agg.columns]

# Session count
station_counts = (
    sessions_master
    .groupby("station_key", dropna=False)
    .size()
    .reset_index(name="sessions_count")
)
station_agg = station_agg.merge(station_counts, on="station_key", how="left")

# Popular vehicle model
if "vehicle_model" in sessions_master.columns:
    pop_model = (
        sessions_master
        .groupby("station_key")["vehicle_model"]
        .agg(series_mode)
        .reset_index()
        .rename(columns={"vehicle_model": "popular_vehicle_model"})
    )
    station_agg = station_agg.merge(pop_model, on="station_key", how="left")

# Peak time of day
if "time_of_day" in sessions_master.columns:
    peak_time = (
        sessions_master
        .groupby("station_key")["time_of_day"]
        .agg(series_mode)
        .reset_index()
        .rename(columns={"time_of_day": "peak_time_of_day"})
    )
    station_agg = station_agg.merge(peak_time, on="station_key", how="left")

# Peak day of week
if "day_of_week" in sessions_master.columns:
    peak_day = (
        sessions_master
        .groupby("station_key")["day_of_week"]
        .agg(series_mode)
        .reset_index()
        .rename(columns={"day_of_week": "peak_day_of_week"})
    )
    station_agg = station_agg.merge(peak_day, on="station_key", how="left")

# Attach geo fields based on available columns
geo_cols = [c for c in ["latitude", "longitude", "city", "state", "charging_station_location"] if c in sessions_master.columns]
if geo_cols:
    geo_frame = (
        sessions_master[["station_key"] + geo_cols]
        .drop_duplicates("station_key")
    )
    matching_keys = set(station_agg["station_key"]).intersection(set(geo_frame["station_key"]))
    if not geo_frame.empty and len(matching_keys) > 0:
        station_agg = station_agg.merge(geo_frame, on="station_key", how="left")

# Density feature: stations within 2 km
if "latitude" in station_agg.columns and "longitude" in station_agg.columns:
    lat = station_agg["latitude"].values
    lon = station_agg["longitude"].values
    within2 = []
    for i in range(len(station_agg)):
        if pd.isna(lat[i]) or pd.isna(lon[i]):
            within2.append(np.nan)
            continue
        dists = haversine_km(lat[i], lon[i], lat, lon)
        within2.append(int(((dists > 0) & (dists <= 2)).sum()))
    station_agg["stations_within_2km"] = within2

# Unmatched report: entries without latitude
if "latitude" in sessions_master.columns:
    unmatched = sessions_master[sessions_master["latitude"].isna()]
else:
    unmatched = sessions_master.iloc[0:0].copy()
unmatched = unmatched[[c for c in ["station_name", "charging_station_operator", "station_key"] if c in unmatched.columns]].drop_duplicates()

# Save output files
station_agg.to_csv("station_aggregates.csv", index=False)
unmatched.to_csv("unmatched_stations.csv", index=False)

# Display results
station_agg_shape = station_agg.shape
unmatched_shape = unmatched.shape

station_agg_shape, unmatched_shape


((236, 17), (0, 3))